In [ ]:
#ORIE 5750: Applied Machine Learning
#Midterm Project
#Results Notebook
#Samantha Taylor (slt84), Alexandra Magnusson (am3492), Victoria Piroian (vp324)

In [ ]:
'''
# e.g. if using google colab import drive, uncomment lines below
from google.colab import drive
drive.mount('/content/drive')
'''

"\n# e.g. if using google colab import drive, uncomment lines below\nfrom google.colab import drive\ndrive.mount('/content/drive')\n"

In [ ]:
import numpy as np
import pandas as pd
import string
import re
import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt')
nltk.download('wordnet')
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score
from nltk.tokenize import word_tokenize
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report
import xgboost as xgb
from xgboost import XGBClassifier

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/samanthataylor/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/samanthataylor/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/samanthataylor/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/samanthataylor/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/samanthataylor/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/samanthataylor/nltk_data...
[nltk_data]   Package wordnet is already up-t

## Download the data

In [ ]:
alex_path = '/content/drive/My Drive/ORIE 5750 - Applied Machine Learning/Midterm/'
sam_path = '/Users/samanthataylor/Library/CloudStorage/OneDrive-Personal/Documents/Cornell Tech/Fall 2025/ORIE 5750 - Applied ML/midterm/final notebooks/'

train_data = pd.read_csv('train.csv')
val_data = pd.read_csv('val.csv')
test_data = pd.read_csv('test.csv')

X_train = train_data['Phrase']
y_train = train_data['Sentiment']

# get only labelled train data
mask = (y_train != -100)
train_data_clean = train_data[mask]
X_train_clean = X_train[mask]
y_train_clean = y_train[mask]

# get val data
X_val    = val_data['Phrase']
y_val    = val_data['Sentiment']

# get test data
X_test     = test_data['Phrase']


# Define Preprocessing Helper Functions

Our preprocessing function takes a text string and peforms:
    1. Lowercase
    2. Remove punctuation
    3. Remove numbers
    4. Tokenize
    5. Remove stopwords
    6. Lemmatize
    7. Re-join tokens

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    if not isinstance(text, str):
        text = str(text)
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\d+', '', text)
    tokens = word_tokenize(text)
    processed_tokens = [
        lemmatizer.lemmatize(word) for word in tokens
        if word not in stop_words and len(word) > 1]
    return ' '.join(processed_tokens)

test_phrase = "This is a test phrase, 123! It's running well."
print(f"Original: {test_phrase}")
print(f"Cleaned:  {preprocess_text(test_phrase)}")

Original: This is a test phrase, 123! It's running well.
Cleaned:  test phrase running well


# Process 1: Bag of Words -> Multinomial NB -> SGD Classifier

In [ ]:
X_train_preproc = train_data['Phrase'].apply(preprocess_text)
X_val_preproc = val_data['Phrase'].apply(preprocess_text)
X_test_preproc = test_data['Phrase'].apply(preprocess_text)

In [ ]:
def bag_of_word(data,  threshold_M):
    vectorizer = CountVectorizer(binary=True, max_features= threshold_M, min_df=3, max_df = 0.9, ngram_range=(1,2))
    vectorizer.fit(X_train_preproc)
    X = vectorizer.transform(data)
    featurized_data = pd.DataFrame(X.toarray(), columns = vectorizer.get_feature_names_out())
    return featurized_data

In [ ]:
# get the featurized data
X_train = bag_of_word(X_train_preproc, 20000)
X_val = bag_of_word(X_val_preproc, 20000)
X_test = bag_of_word(X_test_preproc, 20000)

In [ ]:
#Part 1: MultinomialNB
clf = MultinomialNB(alpha=1e-3)
clf.fit(X_train[mask], y_train[mask])
sk_y = clf.predict(X_train[~mask])

y_train_full = y_train.copy()
y_train_full[~mask]= sk_y


In [ ]:
#Part 2: SGD
#Tried different alphas to see which gives the best accuracy
'''
for alpha in [1, 10, 100, 1e-3, 1e-4, 1e-5]:
    print(f"Alpha: {alpha}")
    clf_sgd = SGDClassifier(loss='hinge', penalty='l2',alpha=alpha, random_state=42)
    clf_sgd.fit(X_train, y_train_full)
    y_pred = clf_sgd.predict(X_val)
    print('Accuracy: ', accuracy_score(y_val, y_pred))
'''

'\nfor alpha in [1, 10, 100, 1e-3, 1e-4, 1e-5]:\n    print(f"Alpha: {alpha}")\n    clf_sgd = SGDClassifier(loss=\'hinge\', penalty=\'l2\',alpha=alpha, random_state=42)\n    clf_sgd.fit(X_train, y_train_full)\n    y_pred = clf_sgd.predict(X_val)\n    print(\'Accuracy: \', accuracy_score(y_val, y_pred))\n'

In [ ]:
#Trying log_loss instead of hinge loss
'''
for alpha in [1e-3, 1e-4, 1e-5, 1e-6]:
    print(f"Alpha: {alpha}")
    clf_sgd = SGDClassifier(loss='log_loss', penalty='l2',alpha=alpha, random_state=42)
    clf_sgd.fit(X_train, y_train_full)
    y_pred = clf_sgd.predict(X_val)
    print('Accuracy: ', accuracy_score(y_val, y_pred))
'''

'\nfor alpha in [1e-3, 1e-4, 1e-5, 1e-6]:\n    print(f"Alpha: {alpha}")\n    clf_sgd = SGDClassifier(loss=\'log_loss\', penalty=\'l2\',alpha=alpha, random_state=42)\n    clf_sgd.fit(X_train, y_train_full)\n    y_pred = clf_sgd.predict(X_val)\n    print(\'Accuracy: \', accuracy_score(y_val, y_pred))\n'

In [ ]:
#final model: alpha = 1e-4, loss = 'hinge'
clf_sgd = SGDClassifier(loss='hinge', penalty='l2',alpha=1e-4, random_state=42)
clf_sgd.fit(X_train, y_train_full)
y_pred = clf_sgd.predict(X_val)
print('Accuracy: ', accuracy_score(y_val, y_pred))

Accuracy:  0.9208806329549364


In [ ]:
print(classification_report(y_val, y_pred, target_names=['Sentiment 0', 'Sentiment 1', 'Sentiment 2', 'Sentiment 3', 'Sentiment 4']))

              precision    recall  f1-score   support

 Sentiment 0       0.95      0.92      0.93      4564
 Sentiment 1       0.87      0.87      0.87      4714
 Sentiment 2       0.99      0.98      0.99      4748
 Sentiment 3       0.88      0.88      0.88      4606
 Sentiment 4       0.91      0.95      0.93      4624

    accuracy                           0.92     23256
   macro avg       0.92      0.92      0.92     23256
weighted avg       0.92      0.92      0.92     23256



In [ ]:
'''
test_pred = clf.predict(X_test)

df = pd.DataFrame(range(len(test_pred)), columns=['PhraseID'])
df['Sentiment'] = test_pred
df.to_csv('nb_submission_2.csv', index=False)
'''

"\ntest_pred = clf.predict(X_test)\n\ndf = pd.DataFrame(range(len(test_pred)), columns=['PhraseID'])\ndf['Sentiment'] = test_pred\ndf.to_csv('nb_submission_2.csv', index=False)\n"

# Process 2: TF-IDF -> Pseudo Labeling with Logistic Regression-> Ensemble Methods

In [ ]:
# Separate Labeled and Unlabeled Data
train_labeled = train_data[mask].copy()
train_unlabeled = train_data[~mask].copy()

print(f"Labeled train set shape:   {train_labeled.shape}")
print(f"Unlabeled train set shape: {train_unlabeled.shape}")

y_train_labeled = train_labeled['Sentiment']
y_val = val_data['Sentiment']

print(train_labeled['Sentiment'].value_counts(normalize=True).sort_index())

Labeled train set shape:   (24758, 2)
Unlabeled train set shape: (34948, 2)
Sentiment
0    0.200784
1    0.215849
2    0.128565
3    0.231440
4    0.223362
Name: proportion, dtype: float64


In [ ]:
train_labeled['clean_Phrase'] = train_labeled['Phrase'].fillna('').astype(str)
train_unlabeled['clean_Phrase'] = train_unlabeled['Phrase'].fillna('').astype(str)
val_data['clean_Phrase'] = val_data['Phrase'].fillna('').astype(str)
test_data['clean_Phrase'] = test_data['Phrase'].fillna('').astype(str)

In [ ]:
#Part 1
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=3, max_df=0.9)

X_train_tfidf = tfidf_vectorizer.fit_transform(train_labeled['clean_Phrase'])
X_val_tfidf = tfidf_vectorizer.transform(val_data['clean_Phrase'])
X_test_tfidf = tfidf_vectorizer.transform(test_data['clean_Phrase'])

print(f"Shape of X_train_tfidf: {X_train_tfidf.shape}")
print(f"Shape of X_val_tfidf:   {X_val_tfidf.shape}")

Shape of X_train_tfidf: (24758, 20000)
Shape of X_val_tfidf:   (23256, 20000)


In [ ]:
# Pseudo-Labeling
lr_baseline = LogisticRegression(multi_class='multinomial', solver='saga', C=1.0, penalty='l2', max_iter=100, random_state=42, n_jobs=-1)

lr_baseline.fit(X_train_tfidf, y_train_labeled)

X_unlabeled_tfidf = tfidf_vectorizer.transform(train_unlabeled['clean_Phrase'])

unlabeled_probs = lr_baseline.predict_proba(X_unlabeled_tfidf)

top_confidence = unlabeled_probs.max(axis=1)
predicted_labels = unlabeled_probs.argmax(axis=1)

train_unlabeled['pseudo_Sentiment'] = predicted_labels
train_unlabeled['confidence'] = top_confidence


/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [ ]:
# Filtering for High-Confidence Labels
conf_thresh = 0.9

pseudo_labeled_data = train_unlabeled[train_unlabeled['confidence'] > conf_thresh].copy()

pseudo_labeled_data = pseudo_labeled_data.drop(columns=['confidence'])
pseudo_labeled_data = pseudo_labeled_data.rename(columns={'pseudo_Sentiment': 'Sentiment'})

In [ ]:
# Creating Augmented Training Set
X_labeled = train_labeled['clean_Phrase'].to_numpy()
y_labeled = train_labeled['Sentiment'].to_numpy()

high_confidence_mask = (train_unlabeled['confidence'] > conf_thresh)
X_pseudo = train_unlabeled.loc[high_confidence_mask, 'clean_Phrase'].to_numpy()
y_pseudo = train_unlabeled.loc[high_confidence_mask, 'pseudo_Sentiment'].to_numpy()

# Concatenate
if len(X_pseudo) > 0 and len(X_pseudo) == len(y_pseudo):
    X_train_aug = np.concatenate((X_labeled.ravel(), X_pseudo.ravel()))
    y_train_aug = np.concatenate((y_labeled.ravel(), y_pseudo.ravel()))
else:
    X_train_aug = X_labeled
    y_train_aug = y_labeled

print(f"Final X_train_aug (text) shape: {X_train_aug.shape}")
print(f"Final y_train_aug (labels) shape: {y_train_aug.shape}")

Final X_train_aug (text) shape: (41363,)
Final y_train_aug (labels) shape: (41363,)


In [ ]:
# Re-Training on Augmented Data
tfidf_vectorizer_aug = TfidfVectorizer(ngram_range=(1, 2), max_features=20000, min_df=3, max_df=0.9)

X_train_aug_tfidf = tfidf_vectorizer_aug.fit_transform(X_train_aug)
X_val_aug_tfidf = tfidf_vectorizer_aug.transform(val_data['clean_Phrase'])

print(f"Shape of X_train_aug_tfidf: {X_train_aug_tfidf.shape}")
print(f"Shape of X_val_aug_tfidf:   {X_val_aug_tfidf.shape}")

Shape of X_train_aug_tfidf: (41363, 20000)
Shape of X_val_aug_tfidf:   (23256, 20000)


In [ ]:
# Fitting Char TF-IDF vectorizer
char_tfidf_vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=25000, min_df=3, max_df=0.9)

X_train_char_tfidf = char_tfidf_vectorizer.fit_transform(X_train_aug)
X_val_char_tfidf = char_tfidf_vectorizer.transform(val_data['clean_Phrase'])

In [ ]:
# Ensemble Methods
lr_augmented = LogisticRegression(multi_class='multinomial', solver='saga', C=1.0, penalty='l2', max_iter=100, random_state=42, n_jobs=-1)

lr_augmented.fit(X_train_aug_tfidf, y_train_aug)
y_val_pred_augmented = lr_augmented.predict(X_val_aug_tfidf)

augmented_accuracy = accuracy_score(y_val, y_val_pred_augmented)

target_names = ['Sentiment 0', 'Sentiment 1', 'Sentiment 2', 'Sentiment 3', 'Sentiment 4']
print(classification_report(y_val, y_val_pred_augmented, target_names=target_names))

# Complement Naive Bayes model
cnb_word_tfidf = ComplementNB()
cnb_word_tfidf.fit(X_train_aug_tfidf, y_train_aug)
y_pred_cnb_tfidf = cnb_word_tfidf.predict(X_val_aug_tfidf)
acc_cnb_tfidf = accuracy_score(y_val, y_pred_cnb_tfidf)
print(f"  Accuracy (CNB on Word TF-IDF): {acc_cnb_tfidf * 100:.2f}%")

# SVM on Word TF-IDF
svm_word_tfidf = SGDClassifier(loss='modified_huber', penalty='l2', alpha=1e-4, random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)
svm_word_tfidf.fit(X_train_aug_tfidf, y_train_aug)
y_pred_svm_tfidf = svm_word_tfidf.predict(X_val_aug_tfidf)
acc_svm_tfidf = accuracy_score(y_val, y_pred_svm_tfidf)
print(f"  Accuracy (SVM on Word TF-IDF): {acc_svm_tfidf * 100:.2f}%")

# SVM on Char TF-IDF
svm_char_tfidf = SGDClassifier(loss='modified_huber', penalty='l2', alpha=1e-4, random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)
svm_char_tfidf.fit(X_train_char_tfidf, y_train_aug)
y_pred_svm_char = svm_char_tfidf.predict(X_val_char_tfidf)
acc_svm_char = accuracy_score(y_val, y_pred_svm_char)
print(f"  Accuracy (SVM on Char TF-IDF): {acc_svm_char * 100:.2f}%")

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


              precision    recall  f1-score   support

 Sentiment 0       0.95      0.94      0.94      4564
 Sentiment 1       0.90      0.87      0.89      4714
 Sentiment 2       0.99      0.99      0.99      4748
 Sentiment 3       0.89      0.91      0.90      4606
 Sentiment 4       0.93      0.95      0.94      4624

    accuracy                           0.93     23256
   macro avg       0.93      0.93      0.93     23256
weighted avg       0.93      0.93      0.93     23256

  Accuracy (CNB on Word TF-IDF): 92.69%
  Accuracy (SVM on Word TF-IDF): 93.95%
  Accuracy (SVM on Char TF-IDF): 93.97%


In [ ]:
# Re-initialize the models so they are fresh for cross_val_predict
model_lr = LogisticRegression(multi_class='multinomial', solver='saga', C=1.0, penalty='l2', max_iter=100, random_state=42, n_jobs=-1)
oof_preds_lr = cross_val_predict(model_lr, X_train_aug_tfidf, y_train_aug, cv=5, method='predict_proba', n_jobs=-1)

model_svm_tfidf = SGDClassifier(loss='modified_huber', penalty='l2', alpha=1e-4, random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)
oof_preds_svm_tfidf = cross_val_predict(model_svm_tfidf, X_train_aug_tfidf, y_train_aug, cv=5, method='predict_proba', n_jobs=-1)

model_cnb_tfidf = ComplementNB()
oof_preds_cnb_tfidf = cross_val_predict(model_cnb_tfidf, X_train_aug_tfidf, y_train_aug, cv=5, method='predict_proba', n_jobs=-1)

model_svm_char = SGDClassifier(loss='modified_huber', penalty='l2', alpha=1e-4, random_state=42, max_iter=1000, tol=1e-3, n_jobs=-1)
oof_preds_svm_char = xx(model_svm_char, X_train_char_tfidf, y_train_aug, cv=5, method='predict_proba', n_jobs=-1)

X_train_meta = np.concatenate(
    (oof_preds_lr, oof_preds_svm_tfidf, oof_preds_cnb_tfidf, oof_preds_svm_char), axis=1)

print(f"Shape of X_train_meta (Level 1 training features): {X_train_meta.shape}")
print(f"Shape of y_train_aug (Level 1 training labels): {y_train_aug.shape}")

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in ve

Shape of X_train_meta (Level 1 training features): (41363, 20)
Shape of y_train_aug (Level 1 training labels): (41363,)


In [ ]:
val_preds_lr = lr_augmented.predict_proba(X_val_aug_tfidf)
val_preds_svm_tfidf = svm_word_tfidf.predict_proba(X_val_aug_tfidf)
val_preds_cnb_tfidf = cnb_word_tfidf.predict_proba(X_val_aug_tfidf)
val_preds_svm_char = svm_char_tfidf.predict_proba(X_val_char_tfidf)

# concat predictions
X_val_meta = np.concatenate(
    (val_preds_lr, val_preds_svm_tfidf, val_preds_cnb_tfidf, val_preds_svm_char),
    axis=1)

# XGBoost
xgb_meta_model = xgb.XGBClassifier(objective='multi:softmax', num_class=5, n_estimators=80, learning_rate=0.1, max_depth=3, subsample=0.7, colsample_bytree=0.7, random_state=42, n_jobs=-1)
xgb_meta_model.fit(X_train_meta, y_train_aug)
y_val_pred_stacked = xgb_meta_model.predict(X_val_meta)

stacked_accuracy = accuracy_score(y_val, y_val_pred_stacked)

target_names = ['Sentiment 0', 'Sentiment 1', 'Sentiment 2', 'Sentiment 3', 'Sentiment 4']
print(classification_report(y_val, y_val_pred_stacked, target_names=target_names))

              precision    recall  f1-score   support

 Sentiment 0       0.96      0.96      0.96      4564
 Sentiment 1       0.91      0.89      0.90      4714
 Sentiment 2       1.00      0.99      1.00      4748
 Sentiment 3       0.91      0.91      0.91      4606
 Sentiment 4       0.95      0.97      0.96      4624

    accuracy                           0.94     23256
   macro avg       0.94      0.94      0.94     23256
weighted avg       0.94      0.94      0.94     23256



In [ ]:
X_test_aug_tfidf = tfidf_vectorizer_aug.transform(test_data['clean_Phrase'])
X_test_char_tfidf = char_tfidf_vectorizer.transform(test_data['clean_Phrase'])

print(f"Shape of X_test_aug_tfidf: {X_test_aug_tfidf.shape}")
print(f"Shape of X_test_char_tfidf: {X_test_char_tfidf.shape}")

Shape of X_test_aug_tfidf: (23257, 20000)
Shape of X_test_char_tfidf: (23257, 25000)


In [ ]:
test_preds_lr = lr_augmented.predict_proba(X_test_aug_tfidf)
test_preds_svm_tfidf = svm_word_tfidf.predict_proba(X_test_aug_tfidf)
test_preds_cnb_tfidf = cnb_word_tfidf.predict_proba(X_test_aug_tfidf)
test_preds_svm_char = svm_char_tfidf.predict_proba(X_test_char_tfidf)

X_test_meta = np.concatenate(
    (test_preds_lr, test_preds_svm_tfidf, test_preds_cnb_tfidf, test_preds_svm_char),
    axis=1
)

In [ ]:
test_pred = xgb_meta_model.predict(X_test_meta)

df = pd.DataFrame(range(len(test_pred)), columns=['PhraseID'])
df['Sentiment'] = test_pred
#df.to_csv('xgboost_submission.csv', index=False)